In [1]:
import os
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv(dotenv_path=".env", override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing from .env")

llm = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=0,
    max_tokens=500,
)

In [ ]:
# response = llm.invoke("What is AI? Max 300 words.")
# print(response.content)

## **RAG IMPLEMENTATION with PDF data**

#### **Step 1: Extracting Text from PDF**

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/fabric-admin.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()
docs

/tmp/ipykernel_736248/2321003938.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26193.03', 'creator': 'Microsoft Learn', 'creationdate': '2026-09-04T16:57:39+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2026-09-04T16:57:39+00:00', 'source': './Docs/fabric-admin.pdf', 'total_pages': 323, 'page': 0, 'page_label': '1'}, page_content='Fabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅ OVERVIEW\nWhat is administration in Fabric?\nｂ GET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃ HOW-TO GUIDE\nUnderstand Fabric admin roles\nｉ REFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅ OVERVIEW\nAbout tenant settings\nｃ HOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26193.03', 'creator': 'Microsoft Learn', 'creationdate

#### **STEP 1.1:(Optional) METADATA creation**

In [3]:
x=0
for i in docs:
    i.metadata = {
        "source":"fabric-admin.pdf",
        "developer":"Microsoft",
        "page":f"{x}"
        }
    x = x+1

#### **Step 2: CHUNKING**

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
len(chunks)

591

In [5]:
chunks[0].metadata

{'source': 'fabric-admin.pdf', 'developer': 'Microsoft', 'page': '0'}

#### **STEP 3: Configure EMBEDDING MODEL**

In [6]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="qwen/qwen3-embedding-8b",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    # IMPORTANT for OpenRouter
    check_embedding_ctx_length=False,
    # Recommended for OpenRouter-compatible providers
    # encoding_format="float",
)

#### **Step 4: Store embeddings in EXISTING LOCAL Vector store**

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="./VectorStore/",
    embedding_function=embedding_model
)

vectorstore.add_documents(chunks)

['630dd90e-f5fa-4266-a993-a94aab03f158',
 '0579254e-7188-48cd-9574-79107c6cd7b3',
 '210fcdb9-5f83-4dee-b0af-29473cd7dd8d',
 '04bbc109-cc52-4082-988e-0850c24523cc',
 '98a361d3-6804-4675-ac8e-81323e3b9471',
 'cf76d705-a58c-4227-a0b7-b86664a6d3cb',
 '9e2c9508-3048-4a18-8b39-fa962a5474dc',
 '9539b7ac-d1a8-4065-a57b-9db9d8c8596a',
 'cd97a65e-7065-4ac3-8592-881088b84819',
 '9922ba74-3a1c-4ef6-aa0d-da1302cd0750',
 '24a13055-7b1f-4d30-9cea-e3b397dd7f4b',
 '8344569c-3d25-4c3f-9bb0-c16b152ca633',
 '34435d9e-79ca-4144-a201-172f776fde56',
 '5ed6cc1c-bf27-4175-810a-40302cd33407',
 '5baab933-8fc7-4f7f-aae7-38ae1bd644e7',
 '1b24219e-6172-43a3-8cfe-d18f7e8313f2',
 '03edd97d-ae60-42cf-bca9-2b5a9785ec38',
 'ce181c48-2f3f-41d8-886e-a11acf8e2497',
 '3be1a216-9820-40b6-84de-a49c9e119e83',
 'b106d140-dab3-42fe-b357-f265c81cb395',
 'acd3780e-2f60-4dd1-9eff-e2f96a079b4e',
 '327a60c2-2672-4541-bb89-6733af5912d1',
 'f368132c-a469-44cc-befd-a1123be7ec9a',
 'ce21c3d9-494f-4b7f-96d3-bc15f78adc33',
 'ba86acc1-72b3-

#### **Step 5: SEMANTIC SEARCH** 

In [9]:
context = vectorstore.similarity_search("What is Microsoft Fabric Admin?",k=3)
context

[Document(id='210fcdb9-5f83-4dee-b0af-29473cd7dd8d', metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft', 'page': '2'}, page_content='What is Microsoft Fabric administration?\nFabric administration is the set of tasks and tools you use to configure, secure, and govern the\nFabric software as a service (SaaS) platform across your organization. As an admin, you control\ntenant-wide settings, manage feature access to meet company policies and regulations, and\ndelegate responsibilities so no single team becomes a bottleneck.\nThere are generally three categories of tasks that admins focus on to ensure the platform is\nconfigured correctly and compliant with organizational policies:\nAdministration—This Fabric administration documentation covers how to manage the\nFabric platform, configure tenant and workspace features, and monitor usage and activity.\nSecurity—See the Security documentation to learn how to help safeguard data with identity,\naccess, encryption, and network p

#### **LETS TALK TO LLM FINALLY**

In [ ]:
response_context = llm.invoke(f"How is AI being used in research fields?? You can answer using the following context: {context}")
print(response_context.content)